In [0]:
silver_df = spark.table("silver_nba_shot_logs_cleaned")

display(silver_df.limit(5))

In [0]:
from pyspark.sql.functions import col, when

gold_df = (
    silver_df
    .withColumn("is_three_pointer", when(col("pts_type") == 3, 1).otherwise(0))
    .withColumn("is_late_clock", when(col("shot_clock") <= 5, 1).otherwise(0))
    .withColumn("is_very_tight_defense", when(col("close_def_dist") <= 2, 1).otherwise(0))
    .withColumn("is_tight_defense", when((col("close_def_dist") > 2) & (col("close_def_dist") <= 4), 1).otherwise(0))
    .withColumn("is_open_shot", when((col("close_def_dist") > 4) & (col("close_def_dist") <= 6), 1).otherwise(0))
    .withColumn("is_wide_open_shot", when(col("close_def_dist") > 6, 1).otherwise(0))
    .withColumn("is_close_shot", when(col("shot_dist") <= 10, 1).otherwise(0))
    .withColumn("is_mid_range", when((col("shot_dist") > 10) & (col("shot_dist") < 23.75), 1).otherwise(0))
    # .withColumn("is_three_point_range", when(col("shot_dist") >= 23.75, 1).otherwise(0))
    .withColumn("is_deep_three", when(col("shot_dist") >= 28, 1).otherwise(0))
)

In [0]:
feature_cols = [
    "game_id",
    "player_name",
    "player_id",
    "period",
    "shot_clock",
    "dribbles",
    "touch_time",
    "shot_dist",
    "pts_type",
    "close_def_dist",
    "is_three_pointer",
    "is_late_clock",
    "is_very_tight_defense",
    "is_tight_defense",
    "is_open_shot",
    "is_wide_open_shot",
    "is_close_shot",
    "is_mid_range",
    "is_deep_three",
    "label"
]

gold_model_df = gold_df.select(*feature_cols)

display(gold_model_df.limit(10))

In [0]:
spark.sql("DROP TABLE IF EXISTS gold_nba_shot_quality_features")
gold_model_df.write.mode("overwrite").saveAsTable("gold_nba_shot_quality_features")

In [0]:
spark.sql("""
SELECT 
  COUNT(*) AS rows,
  AVG(label) AS make_rate,
  AVG(shot_dist) AS avg_shot_dist,
  AVG(close_def_dist) AS avg_close_def_dist,
  AVG(is_three_pointer) AS three_point_rate,
  AVG(is_late_clock) AS late_clock_rate,
  AVG(is_very_tight_defense) AS very_tight_defense_rate,
  AVG(is_tight_defense) AS tight_defense_rate,
  AVG(is_open_shot) AS open_shot_rate,
  AVG(is_wide_open_shot) AS wide_open_shot_rate,
  AVG(is_close_shot) AS close_shot_rate,
  AVG(is_mid_range) AS mid_range_rate,
  AVG(is_deep_three) AS deep_three_rate
FROM gold_nba_shot_quality_features
""").show()

In [0]:
spark.sql("""
SELECT 
  is_three_pointer,
  COUNT(*) AS shots,
  AVG(label) AS make_rate,
  AVG(shot_dist) AS avg_shot_distance,
  AVG(close_def_dist) AS avg_defender_distance
FROM gold_nba_shot_quality_features
GROUP BY is_three_pointer
ORDER BY is_three_pointer
""").show()

In [0]:
spark.sql("""
SELECT 
  CASE 
    WHEN close_def_dist < 2 THEN 'Very Tight: <2 ft'
    WHEN close_def_dist >= 2 AND close_def_dist < 4 THEN 'Tight: 2-4 ft'
    WHEN close_def_dist >= 4 AND close_def_dist < 6 THEN 'Open: 4-6 ft'
    ELSE 'Wide Open: 6+ ft'
  END AS defender_distance_bucket,
  COUNT(*) AS shots,
  AVG(label) AS make_rate
FROM gold_nba_shot_quality_features
GROUP BY 
  CASE 
    WHEN close_def_dist < 2 THEN 'Very Tight: <2 ft'
    WHEN close_def_dist >= 2 AND close_def_dist < 4 THEN 'Tight: 2-4 ft'
    WHEN close_def_dist >= 4 AND close_def_dist < 6 THEN 'Open: 4-6 ft'
    ELSE 'Wide Open: 6+ ft'
  END
ORDER BY make_rate DESC
""").show()